# 🏦 Lesson 07c — Production Multi-Agent Systems
**Domain:** Banking CRM · **Dataset:** Telco Customer Churn
**Duration:** ~2.5 hours

In [ ]:
# ── Imports & Setup ──────────────────────────────────────────────────────────
import os, json, time, base64, datetime
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()  # loads OPENAI_API_KEY and ANTHROPIC_API_KEY from .env

# ─ LLM clients ───────────────────────────────────────────────────
from openai import OpenAI
from anthropic import Anthropic

openai_client  = OpenAI()
claude_client  = Anthropic()

# ─ LM Studio (free local inference) ──────────────────────────────
lm_client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")

# ─ Course checker ────────────────────────────────────────────────
from llm_checker import check, hint, evaluate, progress, show_exercise

print("✅  All imports OK — Module 7 ready")

---
## Lesson 07c — Production Multi-Agent Systems (~2.5 h)
### Orchestrator + Specialist Nodes with Guardrails and Audit Logging

**Learning objectives:**
- Build a LangGraph orchestrator that routes to specialist nodes
- Implement shared state across agents
- Add guardrails: output validation, block low-risk actions
- Human-in-the-loop interrupts for high-value actions
- Audit log: every decision timestamped with agent, action, rationale


In [ ]:
# ── Shared state schema ───────────────────────────────────────────
from typing import TypedDict, Optional, List, Any
from pydantic import BaseModel

class AuditEntry(BaseModel):
    ts:      str
    agent:   str
    action:  str
    summary: str
    blocked: bool = False

class CRMState(TypedDict):
    customer_id:     str
    query:           str
    complaint_text:  str
    churn_proba:     float
    policy_snippet:  str
    offer_draft:     Optional[str]
    offer_value_pln: float
    audit_log:       List[dict]
    route:           str      # "policy" | "churn" | "offer" | "end"

def log_entry(state: CRMState, agent: str, action: str, summary: str, blocked: bool = False) -> None:
    state["audit_log"].append({
        "ts":      datetime.datetime.now().isoformat(timespec="seconds"),
        "agent":   agent,
        "action":  action,
        "summary": summary,
        "blocked": blocked,
    })

print("✅ State schema defined")

In [ ]:
# ── LangGraph orchestrator ─────────────────────────────────────────
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

# ─ Node: Orchestrator ──────────────────────────────────────────
def orchestrator_node(state: CRMState) -> CRMState:
    """Route incoming query to the right specialist node."""
    query = state["query"].lower()
    if any(kw in query for kw in ["policy", "rule", "regulation", "comply"]):
        state["route"] = "policy"
    elif any(kw in query for kw in ["churn", "cancel", "leave", "retention"]):
        state["route"] = "churn"
    elif any(kw in query for kw in ["offer", "discount", "deal", "promo"]):
        state["route"] = "offer"
    else:
        state["route"] = "churn"   # default
    log_entry(state, "OrchestratorNode", "route", f"Routed to: {state['route']}")
    return state

# ─ Node: PolicyRAG ─────────────────────────────────────────────
def policy_rag_node(state: CRMState) -> CRMState:
    """Answer policy questions using LLM (mock RAG for now)."""
    response = lm_client.chat.completions.create(
        model="local-model",
        max_tokens=256,
        messages=[{
            "role": "user",
            "content": f"Bank policy Q&A: {state['query']}\nAnswer in 2 sentences."
        }]
    )
    state["policy_snippet"] = response.choices[0].message.content.strip()
    log_entry(state, "PolicyRAGNode", "retrieve", f"Policy retrieved: {state['policy_snippet'][:80]}...")
    return state

# ─ Node: ChurnAnalysis ─────────────────────────────────────────
def churn_analysis_node(state: CRMState) -> CRMState:
    """Predict churn probability (mock ML for now, replace with real model in capstone)."""
    import random
    random.seed(hash(state["customer_id"]) % 1000)
    state["churn_proba"] = round(random.uniform(0.1, 0.95), 2)
    log_entry(state, "ChurnAnalysisNode", "predict",
              f"churn_proba={state['churn_proba']}")
    return state

# ─ Node: RetentionOffer ────────────────────────────────────────
def retention_offer_node(state: CRMState) -> CRMState:
    """Draft personalized retention offer — ONLY if churn_proba > 0.6."""
    if state["churn_proba"] <= 0.6:
        state["offer_draft"] = None
        log_entry(state, "RetentionOfferNode", "block_offer",
                  f"Blocked: churn_proba={state['churn_proba']} ≤ 0.6", blocked=True)
        return state

    response = lm_client.chat.completions.create(
        model="local-model",
        max_tokens=200,
        messages=[{
            "role": "user",
            "content": (
                f"Write a personalized retention offer for a bank customer "
                f"with churn probability {state['churn_proba']:.0%}. "
                "Keep it to 2 sentences. Suggest a concrete benefit (e.g. waived fee)."
            )
        }]
    )
    draft = response.choices[0].message.content.strip()
    # Extract estimated PLN value (simplified heuristic)
    import re
    nums = [float(x) for x in re.findall(r'\d+\.?\d*', draft)]
    state["offer_value_pln"] = max(nums) if nums else 25.0
    state["offer_draft"]     = draft
    log_entry(state, "RetentionOfferNode", "draft_offer", f"Offer drafted (value={state['offer_value_pln']} PLN)")
    return state

# ─ Routing logic ───────────────────────────────────────────────
def route_decision(state: CRMState) -> str:
    return state["route"]

# ─ Build graph ─────────────────────────────────────────────────
builder = StateGraph(CRMState)
builder.add_node("orchestrator",    orchestrator_node)
builder.add_node("policy_rag",      policy_rag_node)
builder.add_node("churn_analysis",  churn_analysis_node)
builder.add_node("retention_offer", retention_offer_node)

builder.set_entry_point("orchestrator")
builder.add_conditional_edges("orchestrator", route_decision, {
    "policy": "policy_rag",
    "churn":  "churn_analysis",
    "offer":  "retention_offer",
})
builder.add_edge("policy_rag",      END)
builder.add_edge("churn_analysis",  "retention_offer")
builder.add_edge("retention_offer", END)

memory  = MemorySaver()
crm_app = builder.compile(checkpointer=memory)
print("✅ LangGraph CRM orchestrator compiled")

In [ ]:
# ── Test the orchestrator with 3 query types ──────────────────────
def run_crm(customer_id: str, query: str, complaint: str = "") -> CRMState:
    initial_state: CRMState = {
        "customer_id":     customer_id,
        "query":           query,
        "complaint_text":  complaint,
        "churn_proba":     0.0,
        "policy_snippet":  "",
        "offer_draft":     None,
        "offer_value_pln": 0.0,
        "audit_log":       [],
        "route":           "",
    }
    config = {"configurable": {"thread_id": customer_id}}
    final = crm_app.invoke(initial_state, config)
    return final

# Test 3 different query types
test_queries = [
    ("C001", "What is the policy for credit card dispute resolution?"),
    ("C002", "This customer is considering cancelling their account"),
    ("C003", "Draft a retention offer for this loyal customer"),
]

for cid, query in test_queries:
    result = run_crm(cid, query)
    print(f"\n👤 {cid} | route={result['route']} | churn={result['churn_proba']:.2f}")
    print(f"   offer={'✅' if result['offer_draft'] else '❌ blocked'}")
    for entry in result["audit_log"]:
        blocked = " 🚫" if entry["blocked"] else ""
        print(f"   [{entry['agent']}] {entry['action']}{blocked}")

---
### 🟡 EXERCISE Ex 07c-1 — Orchestrator + 3 Specialist Nodes

Build the full orchestrator routing to three nodes. Verify routing works for all 3 query types and audit log is populated correctly.

### 🔴 CHALLENGE Ex 07c-2 — Guardrails for Retention Offers

Extend `RetentionOfferNode`:
1. **Guardrail:** block offer if `churn_probability ≤ 0.6`
2. **Human interrupt:** if `offer_value > 50 PLN`, pause for approval
3. **Log** every SEND/BLOCK decision with reason

**Test:** `churn_prob=0.4` → offer must be blocked + logged


In [ ]:
show_exercise(
    "07c-1", "Orchestrator + 3 Specialist Nodes",
    "Implement the full LangGraph CRM pipeline. Routing must work for policy/churn/offer queries. "
    "Audit log must have ≥ 1 entry per node execution.",
    checks=[
        "Routing works for 3 different query types",
        "audit_log contains ≥ 1 entry after each node",
        "Shared state correctly passed between nodes"
    ]
)

In [ ]:
# ── Auto-check Ex 07c-1 ───────────────────────────────────────────
policy_result = run_crm("T001", "What is the policy for late payment fees?")
churn_result  = run_crm("T002", "Customer wants to cancel — churn risk")
offer_result  = run_crm("T003", "Draft a retention offer for high-value customer")

check([
    (policy_result["route"] == "policy",   "Policy query → policy_rag node"),
    (churn_result["route"]  == "churn",    "Churn query  → churn_analysis node"),
    (offer_result["route"]  == "offer",    "Offer query  → retention_offer node"),
    (len(policy_result["audit_log"]) >= 1, "audit_log populated for policy query"),
    (len(churn_result["audit_log"])  >= 2, "audit_log populated for churn query (≥2 nodes)"),
    (len(offer_result["audit_log"])  >= 2, "audit_log populated for offer query (≥2 nodes)"),
], exercise_id="07c-1")

In [ ]:
show_exercise(
    "07c-2", "Guardrails for Retention Offers",
    "Add guardrail: block offer if churn_prob ≤ 0.6. "
    "Human interrupt if offer_value > 50 PLN. Log every SEND/BLOCK decision.",
    hints=[
        "▸ if churn_probability < 0.6: return {'offer_draft': None, 'blocked_reason': 'low_risk'}",
        "▸ graph.add_interrupt_before(['send_offer']) for human approval",
    ],
    checks=[
        "churn_prob ≤ 0.6 → offer blocked and reason logged",
        "offer_value > 50 PLN → human approval requested",
        "Every SEND/BLOCK in audit_log with reason"
    ],
    exercise_type="CHALLENGE"
)

In [ ]:
# ── Auto-check Ex 07c-2 (Guardrails) ─────────────────────────────

# Force churn_prob = 0.4 (should block)
low_risk_state: CRMState = {
    "customer_id": "GUARDRAIL_TEST",
    "query": "Draft offer",
    "complaint_text": "",
    "churn_proba": 0.4,   # below threshold
    "policy_snippet": "",
    "offer_draft": None,
    "offer_value_pln": 0.0,
    "audit_log": [],
    "route": "offer",
}
# Run only the retention_offer_node directly
guardrail_result = retention_offer_node(low_risk_state)

blocked_entries = [e for e in guardrail_result["audit_log"] if e.get("blocked")]

check([
    (guardrail_result["offer_draft"] is None, "churn_prob=0.4 → offer draft is None (blocked)"),
    (len(blocked_entries) >= 1,               "Block decision appears in audit_log"),
    (blocked_entries[0]["agent"] == "RetentionOfferNode" if blocked_entries else False,
     "Block logged by RetentionOfferNode"),
], exercise_id="07c-2")

In [ ]:
# ── Module 7 Progress ─────────────────────────────────────────────
progress()

---
## ✅ Module 7 Readiness Checklist
- ☐ Reasoning models benchmark with cost/quality table (07a-1)
- ☐ Vision extraction pipeline on synthetic statement (07b-1)
- ☐ Orchestrator + 3 nodes with routing (07c-1)
- ☐ Guardrails block low-risk offers (07c-2)
- ☐ Audit log captures every agent decision (07c-2)

**Next:** Module 8 — Capstone Preparation & System Integration
